<a href="https://colab.research.google.com/github/thakur785/sandbox_llm/blob/ai-agents/AIAgents/AIAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#learning AI agents using langgraph

In [1]:
from dotenv import load_dotenv
_ = load_dotenv()

In [2]:
!pip install -q langchain langchain-openai openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 3.0 MB/s eta 0:00:00


In [4]:
!pip install -q langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [5]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

/tmp/ipykernel_3213/2233616850.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [9]:
from google.colab import userdata
import os
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
tool = TavilySearchResults(max_results=1) #increased number of results
print(type(tool))
print(tool.name)

<class 'langchain_community.tools.tavily_search.tool.TavilySearchResults'>
tavily_search_results_json


# AI Agent in langgraph
## Agentic search tool:


In [10]:
!pip install tavily-python

In [11]:
#libraries
from dotenv import load_dotenv
import os
from tavily import TavilyClient
# load environment variable from .env file
_ = load_dotenv()


In [12]:
#connect
client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

In [13]:
#  run search
result = client.search("what is in Nvidia's new Blackwell GPU?", include_answer=True)
#print the answer
result["answer"]

"The new Blackwell GPU in Nvidia's architecture features 208 billion transistors and is made using a custom TSMC 4NP process. It includes advanced streaming multiprocessors and CUDA core technology for enhanced performance."

In [14]:
#weather serach
#query
city = "Bengaluru"
query = f"""
        what is the traffic in {city} in 2026?
"""

In [15]:
# run the search
result = client.search(query, max_results=1)
#print first result

data = result["results"][0]["content"]
print(data)

Never miss a post from supri\_v. Sign up for Instagram to stay in the loop. Bengaluru records the highest traffic of 2026. hopr.mobi's profile picture. @supri\_v highest traffic of 2026 built one solo car at a time 😤 if every corporate employee on this stretch shared a ride, 70% of this jam doesn't exist tomorrow 🚗. ORR is filled with traffic all the days 😟. Why all the company are at the same place. This is what netizens are saying, whole world is in banglore now since years. sudhash.r's profile picture. Please start a rules if a car having one person will not be allowed in High traffic areas. anamul.king_20.07's profile picture. abhisek.bedant's profile picture. Today's update yet to come wait for the HAL traffic update 🔥🔥🔥🔥. Let's see the magic it does. It is instagram's algorithm or some magic. What are your thoughts on the CBSE OSM controversy? RCB is not just a team… it’s an emotion.


## Persistence and streaming


In [16]:
from dotenv import load_dotenv
_ = load_dotenv()

In [17]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

In [18]:
from google.colab import userdata
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [19]:
tool = TavilySearchResults(max_results=2)

In [20]:
class AgentState(TypedDict):
  messages: Annotated[list[AnyMessage], operator.add]

In [21]:
!pip install -q langgraph.checkpoint.sqlite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 6.8 MB/s eta 0:00:00


In [22]:
!pip install -q langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.9 MB/s eta 0:00:00


In [23]:
from langgraph.checkpoint.memory import InMemorySaver
memory = InMemorySaver()

In [24]:
# create agent with persistency
class Agent:
  def __init__(self, model, tools, checkpointer, system=""):
    self.system = system
    graph = StateGraph(AgentState)
    graph.add_node("llm", self.call_openai)
    graph.add_node("action", self.take_action)
    graph.add_conditional_edges("llm", self.exists_action, {True: "action", False:END})
    graph.add_edge("action", "llm")
    graph.set_entry_point("llm")
    self.graph = graph.compile(checkpointer=checkpointer)
    self.tools = {t.name: t for t in tools}
    self.model = model.bind_tools(tools)

  def call_openai(self, state: AgentState):
    messages = state["messages"]
    if self.system:
      messages = [SystemMessage(content=self.system)] + messages
    message = self.model.invoke(messages)
    return {'messages': [message]}

  def exists_action(self, state: AgentState):
    result = state["messages"][-1]
    return len(result.tool_calls) > 0

  def take_action(self, state: AgentState):
    tool_calls = state["messages"][-1].tool_calls
    results = []
    for t in tool_calls:
      print (f"calling: {t}")
      result = self.tools[t['name']].invoke(t['args'])
      results.append(ToolMessage(tool_call_id=t['name'], content=str(result)))
      print("Back to Model!")
      return {'messages': results}

In [25]:
print(type(memory))


<class 'langgraph.checkpoint.memory.InMemorySaver'>


In [26]:
from google.colab import userdata
from langchain_groq import ChatGroq
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatGroq(model="llama-3.3-70b-versatile")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [27]:
messages = [HumanMessage(content="What is the weather in sf?")]

In [28]:
thread = {"configurable": {"thread_id": "1"}}

In [29]:
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v['messages'])

[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'e3p2p9cv5', 'function': {'arguments': '{"query":"San Francisco weather today"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 350, 'total_tokens': 371, 'completion_time': 0.058515593, 'completion_tokens_details': None, 'prompt_time': 0.034263625, 'prompt_tokens_details': None, 'queue_time': 0.036794567, 'total_time': 0.092779218}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_d42c28f9ce', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eaab2-234f-7373-97d5-bdbea8b5bf05-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'San Francisco weather today'}, 'id': 'e3p2p9cv5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 350, 'output_tokens': 21, 'total_tokens': 371})]
calling: {'name': 'tavi

In [30]:
messages = [HumanMessage(content="What about in la?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'jn30wza9g', 'function': {'arguments': '{"query":"Los Angeles weather today"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 1995, 'total_tokens': 2016, 'completion_time': 0.029784474, 'completion_tokens_details': None, 'prompt_time': 0.196972982, 'prompt_tokens_details': None, 'queue_time': 0.059966682, 'total_time': 0.226757456}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_d42c28f9ce', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eaab2-3a41-7ac3-b467-af22d9c881ec-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'Los Angeles weather today'}, 'id': 'jn30wza9g', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1995, 'output_tokens': 21, 'total_tokens': 2016})]}
calling: 

In [31]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='Based on the previous responses, the current temperature in San Francisco is 16°C (61°F), while the current temperature in Los Angeles is not explicitly stated in the search results. However, the search results for Los Angeles provide a 10-day forecast with high temperatures ranging from 22°C to 28°C (72°F to 82°F).\n\nAssuming the current temperature in Los Angeles is around the average high temperature for the day, which is around 24°C (75°F), Los Angeles would be warmer than San Francisco.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 108, 'prompt_tokens': 5935, 'total_tokens': 6043, 'completion_time': 0.407392864, 'completion_tokens_details': None, 'prompt_time': 0.313089272, 'prompt_tokens_details': None, 'queue_time': 0.059830114, 'total_time': 0.720482136}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model

# Streaming Token

In [36]:
!pip install langgraph-checkpoint-sqlite aiosqlite

In [46]:
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
memory = AsyncSqliteSaver.from_conn_string(":memory:")
#model = ChatGroq(model="llama-3.3-70b-versatile")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)


In [45]:
messages = [HumanMessage(content="What is the weather in SF?")]
thread = {"configurable": {"thread_id": "4"}}
async for event in abot.graph.astream_events({"messages": messages}, thread, version="v1"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            # Empty content in the context of OpenAI means
            # that the model is asking for a tool to be invoked.
            # So we only print non-empty content
            print(content, end="|")

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3551: LangChainDeprecationWarning: astream_events version='v1' is deprecated. Use version='v2' or astream instead.
  await eval(code_obj, self.user_global_ns, self.user_ns)


calling: {'name': 'tavily_search_results_json', 'args': {'query': 'San Francisco weather today'}, 'id': 'sx9sn48jt', 'type': 'tool_call'}
Back to Model!
The| current| weather| in| San| Francisco| is| partly| cloudy| with| a| feels|-like| temperature| of| |16|°C|.| There| is| a| |25|%| chance| of| rain|,| and| the| humidity| is| |81|%.| The| wind| speed| is| |19|.|4| km|/h|,| and| the| UV| index| is| |1|.|4|,| which| is| considered| low|.| The| |10|-day| forecast| shows| a| mix| of| sunny| and| cloudy| days|,| with| a| high| of| |22|°C| and| a| low| of| |9|°C|.|